# 02 — Baseline Inference: TinyLlama Before Fine-Tuning

**Project**: LLM Fine-Tuning with QLoRA  
**Notebook**: 2 of 6  
**GPU required**: ✅ Yes — T4 or better  
**Runtime**: ~10 minutes (first run downloads ~600 MB)

---

## What This Notebook Does

Before fine-tuning, we run the **base TinyLlama model** (no training, no adapters) on
5 fixed prompts and save its raw outputs.

This gives us our **baseline** — the "before" state.  
After fine-tuning (notebook 04), we'll run the same prompts again and compare.

## Why This Step Matters

> Without a baseline, you can't measure improvement.
> Running the experiment without this is like weighing yourself only after a diet.

Also: understanding how the **base model fails** at instruction following
tells you exactly what fine-tuning needs to fix.

## Key Concepts This Notebook Introduces
- Loading a quantized model with `BitsAndBytesConfig`
- What a tokenizer does step-by-step
- How `model.generate()` works
- The difference between `do_sample` and greedy decoding
- What `torch.no_grad()` is and why we use it

---
## ⚙️ Cell 1: Environment Setup

In [ ]:
# Install all QLoRA dependencies
!pip install -q -U transformers peft trl datasets bitsandbytes accelerate
print("✅ Dependencies installed")

In [ ]:
import os, sys, json

# Clone project repo
GITHUB_USERNAME = "YOUR_USERNAME"   # ← change this
REPO_NAME = "llm-finetuning-peft"

if not os.path.exists(REPO_NAME):
    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
else:
    !cd {REPO_NAME} && git pull

sys.path.insert(0, f"/content/{REPO_NAME}/src")
os.makedirs(f"/content/{REPO_NAME}/results", exist_ok=True)

print(f"✅ Repo ready. src/ added to Python path.")

---
## 🖥️ Cell 2: Verify GPU

**Before loading any model, always check your GPU.**  
If this cell shows "No GPU detected", go to:  
- **Colab**: Runtime → Change runtime type → T4 GPU → Save  
- **Kaggle**: Settings (right panel) → Accelerator → GPU T4 x2

In [ ]:
import torch

print("🖥️  Hardware Check")
print("-" * 40)

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    total_vram = gpu.total_memory / 1e9
    print(f"  GPU Name    : {gpu.name}")
    print(f"  VRAM        : {total_vram:.1f} GB")
    print(f"  CUDA version: {torch.version.cuda}")
    print(f"  PyTorch     : {torch.__version__}")
    print()
    if total_vram < 10:
        print("⚠️  Less than 10GB VRAM — model loading may fail. Use T4 GPU.")
    else:
        print("✅ GPU looks good! Ready to load the model.")
else:
    print("❌ No GPU detected! Please enable GPU runtime before proceeding.")
    print("   Colab: Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("GPU required for this notebook.")

---
## 🔤 Cell 3: Load the Tokenizer

We always load the tokenizer before the model.  
Let's inspect it closely — tokenizers are often overlooked but critical.

**What is a tokenizer?**  
It converts text ↔ integers. The model only understands numbers, not words.  
Every model has its own vocabulary and tokenizer — they must always be paired.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"📥 Loading tokenizer for: {MODEL_NAME}")
print("   (First run: downloads ~500KB tokenizer files)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\n📊 Tokenizer Stats:")
print(f"   Vocabulary size : {tokenizer.vocab_size:,} unique tokens")
print(f"   Model max length: {tokenizer.model_max_length:,} tokens")
print(f"   EOS token       : '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"   PAD token       : '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

In [ ]:
# ── Tokenization deep-dive — see exactly what the model sees ──────────────
example_text = "What is machine learning?"

# Step 1: Text → Token IDs
token_ids = tokenizer.encode(example_text)
print(f"Text     : '{example_text}'")
print(f"Token IDs: {token_ids}")

# Step 2: Token IDs → Token strings (subwords)
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print(f"Tokens   : {tokens}")

# Step 3: Token IDs → Back to text
decoded = tokenizer.decode(token_ids)
print(f"Decoded  : '{decoded}'")

print(f"\n💡 Notice: '{example_text}' = {len(token_ids)} tokens")
print(f"   Words: {len(example_text.split())}  |  Tokens: {len(token_ids)}")
print(f"   Tokens ≠ words. Some words split into multiple subword tokens.")

---
## 🤖 Cell 4: Load TinyLlama in 4-bit (QLoRA)

Now we load the actual model weights. This downloads ~600 MB the first time
and is cached in Colab for the session.

**Why 4-bit?**  
Even for inference (not training), we load in 4-bit so it fits in T4's 15GB VRAM.
Full float16 would be ~2.2 GB — still fits, but 4-bit is faster to load.

Watch the output — you'll see the model layers loading onto the GPU in real time.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# ── 4-bit quantization config ─────────────────────────────────────────────
# Same config we'll use during training — loading consistently is important
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,   # Compute in fp16 for speed
    bnb_4bit_quant_type="nf4",              # NormalFloat4 — optimal for LLMs
    bnb_4bit_use_double_quant=True,         # Extra compression of quant constants
)

print(f"📥 Loading {MODEL_NAME} in 4-bit...")
print("   First run: downloads ~600 MB (cached after)")
print("   Watch layers load onto GPU...\n")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",       # Automatically use GPU
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
base_model.eval()   # Inference mode: disables dropout

# ── Memory usage report ───────────────────────────────────────────────────
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved() / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\n📊 GPU Memory After Loading:")
print(f"   Model using : {allocated:.2f} GB")
print(f"   Reserved    : {reserved:.2f} GB")
print(f"   Available   : {total - reserved:.2f} GB remaining")
print(f"   Total VRAM  : {total:.2f} GB")
print(f"\n💡 In float16, this model would use ~2.2 GB")
print(f"   In 4-bit (NF4), it uses only {allocated:.2f} GB — that's QLoRA's power!")

In [ ]:
# ── Inspect model architecture ────────────────────────────────────────────
# This shows you the actual transformer layers inside TinyLlama

total_params = sum(p.numel() for p in base_model.parameters())
print(f"🏗️  TinyLlama Architecture Summary")
print(f"-" * 40)
print(f"  Total parameters : {total_params:,}")
print(f"  Layers (blocks)  : {base_model.config.num_hidden_layers}")
print(f"  Hidden size      : {base_model.config.hidden_size}")
print(f"  Attention heads  : {base_model.config.num_attention_heads}")
print(f"  Vocabulary size  : {base_model.config.vocab_size:,}")
print(f"  Context window   : {base_model.config.max_position_embeddings:,} tokens")
print()
print("  Model structure (first transformer block):")
print(base_model.model.layers[0])

---
## 💬 Cell 5: Generate Responses — Understanding `model.generate()`

Before running all 5 test prompts, let's understand what `model.generate()` does step by step.

In [ ]:
# ── Step-by-step generation walkthrough ───────────────────────────────────
test_prompt = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\nGive three tips for staying healthy.\n\n### Response:"
)

print("STEP 1: Tokenize the input prompt")
print("-" * 50)
inputs = tokenizer(test_prompt, return_tensors="pt").to(base_model.device)
input_ids = inputs["input_ids"]
print(f"  Prompt text  : {len(test_prompt)} characters")
print(f"  Token IDs    : shape {input_ids.shape}  → {input_ids.shape[1]} tokens")
print(f"  Device       : {input_ids.device}")

print()
print("STEP 2: Run model.generate()")
print("-" * 50)
print("  - Model predicts next token probabilities")
print("  - Samples from distribution (temperature controls randomness)")
print("  - Appends chosen token, repeats until EOS or max_new_tokens")
print("  This happens ONE TOKEN AT A TIME — that's how all LLMs work!")

input_length = input_ids.shape[1]

with torch.no_grad():   # ← No gradients needed for inference — saves memory
    output_ids = base_model.generate(
        input_ids,
        max_new_tokens=200,
        temperature=0.1,         # Near-deterministic for reproducibility
        do_sample=True,          # Use sampling (respects temperature)
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print()
print("STEP 3: Decode output tokens")
print("-" * 50)
print(f"  Total output tokens : {output_ids.shape[1]}")
print(f"  Input tokens        : {input_length}")
print(f"  New tokens generated: {output_ids.shape[1] - input_length}")

# Slice: only decode NEW tokens (not the prompt)
new_tokens = output_ids[0][input_length:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print()
print("STEP 4: The response")
print("-" * 50)
print(response)

---
## 📋 Cell 6: Run All 5 Baseline Prompts

Now we run all 5 fixed evaluation prompts through the base model and save the outputs.
These exact same prompts will be run again after fine-tuning (in notebook 04).

In [ ]:
from evaluate import QUALITATIVE_PROMPTS, generate_response

print("🔍 Running 5 baseline prompts through the BASE model (no fine-tuning)")
print("=" * 70)
print("Save these outputs mentally — you'll compare them after training!")
print("=" * 70)

baseline_results = []

for item in QUALITATIVE_PROMPTS:
    label  = item["label"]
    prompt = item["prompt"]

    # Extract just the instruction text for display
    instruction = prompt.split("### Instruction:")[1].split("### Response:")[0].strip()

    print(f"\n📌 [{label}]")
    print(f"Instruction: {instruction}")
    print("-" * 70)

    response = generate_response(base_model, tokenizer, prompt, max_new_tokens=200)

    print(f"BASE MODEL OUTPUT:\n{response}")
    print("=" * 70)

    baseline_results.append({
        "label": label,
        "instruction": instruction,
        "prompt": prompt,
        "base_response": response,
    })

print("\n✅ All 5 baseline prompts complete!")

In [ ]:
# ── Save baseline outputs to results/ ─────────────────────────────────────
# These will be loaded in notebook 04 for side-by-side comparison
import json

RESULTS_PATH = f"/content/{REPO_NAME}/results/baseline_outputs.json"

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, indent=2, ensure_ascii=False)

print(f"💾 Baseline outputs saved to: {RESULTS_PATH}")
print(f"   {len(baseline_results)} prompts × responses stored")
print()
print("📌 Commit this file to GitHub so you have the baseline saved permanently:")
print("   git add results/baseline_outputs.json")
print("   git commit -m 'results: add baseline inference outputs'")
print("   git push")

---
## 🧠 Cell 7: Analyze Base Model Behavior

Before moving on, let's think critically about what we observed.

In [ ]:
# ── Things to observe in the base model outputs ───────────────────────────
print("🔍 Base Model Behavior Analysis")
print("=" * 60)
print("""
Common patterns you'll notice in the BASE model outputs:

❌ Format issues:
   - May not follow the numbered list format when asked
   - May continue generating beyond the answer
   - May repeat phrases or loop
   - May answer a different question than asked

❌ Instruction following:
   - The ### Instruction / ### Response format is familiar
   (TinyLlama-Chat was already instruction-tuned on a different dataset)
   - But Alpaca-specific patterns may not be perfect

✅ What the base model already knows:
   - General world knowledge
   - Grammar and language structure
   - Basic reasoning

💡 Key insight:
   Fine-tuning doesn't teach the model NEW knowledge.
   It teaches the model HOW to package and present what it already knows
   in the format and style we want.

After fine-tuning on Alpaca, you should see:
   ✅ More consistent formatting (numbered lists, clear sections)
   ✅ Responses that better match Alpaca's style
   ✅ More concise, focused answers
   ✅ Better stopping (less runaway generation)
""")

---
## ✅ Summary

In this notebook you:

| Step | What happened |
|---|---|
| Verified GPU | Confirmed T4 with 15GB VRAM |
| Loaded tokenizer | Vocabulary of 32,000 tokens, pad=eos workaround |
| Tokenization deep-dive | Saw text → token IDs → back to text |
| Loaded model (4-bit) | ~0.6 GB VRAM vs ~2.2 GB float16 |
| Inspected architecture | 22 layers, 1.1B params, attention heads, context window |
| Understood generate() | One token at a time, temperature, sampling |
| Ran 5 baseline prompts | Saved to `results/baseline_outputs.json` |

### Key PyTorch Concepts Covered
- `torch.cuda.is_available()` — GPU detection
- `return_tensors="pt"` — return PyTorch tensors from tokenizer
- `.to(device)` — move tensors to GPU
- `torch.no_grad()` — disable gradient tracking for inference
- `model.eval()` — set model to inference mode

---
### ▶️ Next: `03_qlora_finetuning.ipynb`
The main event. We apply LoRA adapters and run the actual fine-tuning.  
Training takes ~30–45 minutes on a free T4 GPU.